# Step 43 — Candidate origin association across hypocenter indexes

This step groups source **origins**, not waveforms or arrival picks. It does not associate these groups with step-32 waveform containers. Both source solutions and source line references are retained; a preferred solution is selected by source preference, not recomputed.

## Algorithm and limitations

1. Combine the five indexes from step 40 (`Pinatubo_all.sum` as `hypo40`, `PINAALL.DAT` as `pinaall`, and the later solutions `gold`, `brun` and `pepi`) and sort by UTC time with a stable sort: equal times keep input order (sources in that order, then source line order), so the event IDs are reproducible. No independent clock correction is performed here.
2. Take the earliest unassigned origin as the anchor. Search later unassigned origins within `time_tol` seconds, and include those within `dist_tol` horizontal kilometres of that anchor.
3. Mark members assigned, then repeat. This is greedy anchor-based clustering, not transitive linkage or one-to-one cross-source matching. Multiple origins from the **same source** can be grouped: the files repeat some earthquakes (about 200 in `b-run.sum`, 165 of them identical; about 20 in the HYPO71 files), and those repeats belong together. `one_origin_per_source=True` would instead allow one origin per source per group, which splits such repeats into separate events. Multiple earthquakes in a swarm could be combined if tolerances are too permissive. Membership does not require every pair to be mutually close.
4. Select the preferred origin by `origin_priority`: PEPI (the published locations of Jones et al., 2001), then BRUN (HYPOELLIPSE, 2001), GOLD (HYPOELLIPSE, 1992–93), `Pinatubo_all.sum`, `PINAALL.DAT`. Later reprocessing ranks first; who produced GOLD and BRUN is not documented. Depth and magnitude are neither matching criteria nor quality-ranking criteria. Horizontal distance uses ObsPy's spherical angular-distance conversion, not a 3-D hypocentral distance.

Sequential event/origin IDs depend on input order and rows (they are reproducible for the same inputs); they are not permanent external identifiers. Source-composition reports show both-source/source-only containers, not an independently validated earthquake count. Check multiorigin same-source groups and time/location differences after running.

## Inputs and safeguards

Only rows with parseable times and finite geographic coordinates should enter automatic association. This notebook now checks those prerequisites before invoking the preserved matching algorithm and stops with a message if unresolved records exist. Original reported times and unresolved annotated records remain in their source indexes for review. Explicitly curate a time-resolved input rather than silently dropping or shifting them. Depth/magnitude may remain missing; there is no automatic quality exclusion.

Outputs replace event and origin CSVs plus optional diagnostic CSVs beneath the external output root. Source-file/line references remain, but not every quality/annotation field is copied to the event table: retain source indexes alongside outputs.

## Magnitude policy (11 November 2026)

Preferred magnitude is chosen independently of the preferred origin: use a nonmissing PINAALL value first, otherwise a nonmissing summary-file value, otherwise leave magnitude missing. GOLD, BRUN and PEPI magnitudes are never preferred: they are on other scales (GOLD about 0.6 and HYPO71 about 0.8 units below BRUN; step 47), so they stay on their own origins. Legacy 0.0 values are provisionally interpreted as missing, not as true M0 earthquakes; negative nonzero magnitudes remain valid. `zero_magnitude_is_missing=False` reverses this policy for future review. Source indexes retain their raw reported values; the normalized origin table preserves them in `reported_magnitude` and records a missing-value reason. The event table records magnitude source and origin ID. Multiple candidate magnitudes within a group still require association review; first valid source value wins.

Missing numeric values are empty CSV fields (pandas NaN), not literal strings `None`, `NULL` or zero. Catalogue export omits absent magnitudes. This is a documented working interpretation of archival zeros, not proof that a physical magnitude of exactly zero is impossible. Rerun 43 and then 50 to regenerate outputs; existing catalogues are not modified by changing this notebook.

## 1. Configuration

Review source paths, selected PINAALL variant and output paths before running. Configuration does not execute the workflow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'lib' / 'clock.py').is_file()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from lib.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    hypo40=P['SUMMARY_FILE_40_INDEX'],
    hypo41=P['SUMMARY_FILE_41_INDEX'],
    extra_sources={'gold': P['GOLD_INDEX'], 'brun': P['BRUN_INDEX'], 'pepi': P['PEPI_INDEX']},
    time_tol=float(P['TIME_TOL_S']),
    dist_tol=float(P['DIST_TOL_KM']),
    # Preferred origin: the published locations of Jones et al. (2001), then the latest reprocessing first (decided 23 Nov 2026).
    origin_priority=('pepi', 'brun', 'gold', 'hypo40', 'pinaall'),
    # Magnitudes are on different scales (GOLD about 0.6 and HYPO71 about 0.8 below BRUN, step 47), so only
    # HYPO71 magnitudes are preferred, as in the paper; the others stay on their own origins.
    preferred_magnitude_sources=('pinaall', 'hypo40'),
    one_origin_per_source=False,  # repeats within a file (about 200 in b-run.sum) are the same earthquake
    zero_magnitude_is_missing=True,
    out_event_csv=P['HYPO71_EVENT_INDEX'],
    out_origin_csv=P['HYPO71_ORIGIN_INDEX'],
    emit_diagnostics=True,
    diagnostics_dir=f"{P['QC_DIR']}/step43_hypo_associate_diagnostics",
)
display(SETTINGS)

## 2. Implementation

The complete step-specific workflow is defined below. Definitions alone do not read sources or write products.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
from obspy.geodetics import locations2degrees, degrees2kilometers

# Source label -> origin_id prefix and the file named in reports.
SOURCE_PREFIX = {'hypo40': 'hypo40', 'pinaall': 'pinaall', 'gold': 'gold', 'brun': 'brun', 'pepi': 'pepi'}


def _normalize(label, raw):
    """One source index -> the common origin table (UTC times, provenance, quality where given)."""
    norm = pd.DataFrame({
        "origin_id": raw.index.map(lambda i: f"{SOURCE_PREFIX[label]}_{i}"),
        "origin_time": pd.to_datetime(raw["origin_time"], format="mixed", utc=True),
        "latitude": raw["latitude"].astype(float),
        "longitude": raw["longitude"].astype(float),
        "depth_km": raw["depth_km"].astype(float),
        "magnitude": raw["magnitude"].astype(float),
        "source": label,
        "source_file": raw["source_file"],
        "source_line": raw["source_line"],
    })
    # UTC-run provenance (present only when the indexes were converted with clock_regimes.csv)
    for col in ("raw_origin_time", "clock_regime", "clock_uncertainty_s"):
        norm[col] = raw[col].values if col in raw else None
    # RMS residual and number of readings, where the source gives them (Pinatubo_all.sum, PEPI).
    norm["rms"] = pd.to_numeric(raw["rms"], errors="coerce").values if "rms" in raw else float("nan")
    norm["nass"] = pd.to_numeric(raw["nass"], errors="coerce").values if "nass" in raw else float("nan")
    norm["note"] = raw["note"].values if "note" in raw else None
    valid = (norm['origin_time'].notna() & np.isfinite(norm['latitude'])
             & np.isfinite(norm['longitude']) & norm['latitude'].between(-90, 90)
             & norm['longitude'].between(-180, 180))
    if not valid.all():
        raise ValueError(f'{label}: {int((~valid).sum())} unresolved time/coordinate rows; review and explicitly select valid input before association.')
    return norm


def run(
    *,
    hypo40,
    hypo41,
    extra_sources=None,
    time_tol,
    dist_tol,
    origin_priority=('hypo40', 'pinaall'),
    preferred_magnitude_sources=('pinaall', 'hypo40'),
    one_origin_per_source=False,
    zero_magnitude_is_missing=True,
    out_event_csv,
    out_origin_csv,
    emit_diagnostics=False,
    diagnostics_dir=None,
):
    """Group origins from all source indexes into events; choose a preferred origin and magnitude.

    hypo40 / hypo41: Pinatubo_all.sum and PINAALL.DAT indexes (sources 'hypo40', 'pinaall').
    extra_sources: {label: index CSV} for the later solutions ('gold', 'brun', 'pepi'); a missing file is an error.
    origin_priority: preferred origin = first member from the earliest-listed source (else the first member).
    preferred_magnitude_sources: magnitudes eligible to be preferred, in order; others stay on their origins only.
    one_origin_per_source: True lets a group take at most one origin from each source. Default False: the
    files repeat some earthquakes (b-run.sum lists about 200 twice, 165 identically; the HYPO71 files about
    20), and those repeats belong in the same group.
    """
    sources = {'hypo40': hypo40, 'pinaall': hypo41, **(extra_sources or {})}
    unknown = set(sources) - set(SOURCE_PREFIX)
    if unknown:
        raise ValueError(f'Unknown sources: {sorted(unknown)}')
    for name, listed in (('origin_priority', origin_priority), ('preferred_magnitude_sources', preferred_magnitude_sources)):
        if not set(listed) <= set(SOURCE_PREFIX):
            raise ValueError(f'{name}: unknown sources {sorted(set(listed) - set(SOURCE_PREFIX))}')

    # ------------------------------------------------------------------
    # Load + normalize inputs
    # ------------------------------------------------------------------
    frames = []
    for label, path in sources.items():
        raw = pd.read_csv(path)
        frame = _normalize(label, raw)
        # Step 40 blanks HYPO71 0.00 and keeps the value as written in reported_magnitude.
        reported = raw['reported_magnitude'] if 'reported_magnitude' in raw else raw['magnitude']
        frame['reported_magnitude'] = pd.to_numeric(reported, errors='coerce').values
        frame['magnitude'] = frame['reported_magnitude']
        frame['magnitude_missing_reason'] = ''
        zero = frame['reported_magnitude'].eq(0) & zero_magnitude_is_missing
        frame.loc[zero, 'magnitude'] = float('nan')
        frame.loc[zero, 'magnitude_missing_reason'] = 'reported_zero_treated_as_missing'
        frames.append(frame)

    # Stable sort: origins with the same time keep input order (sources in the order above, then
    # source line order), so event and origin IDs do not change between runs.
    all_origins = (
        pd.concat(frames, ignore_index=True)
        .sort_values("origin_time", kind="stable")
        .reset_index(drop=True)
    )
    times = all_origins["origin_time"].to_numpy()
    lats, lons = all_origins["latitude"].to_numpy(), all_origins["longitude"].to_numpy()
    srcs = all_origins["source"].to_numpy()

    # ------------------------------------------------------------------
    # Event association: greedy, anchored on the earliest unassigned origin
    # ------------------------------------------------------------------
    events: List[Dict] = []
    origins_out: List[Dict] = []
    event_id = 0
    assigned = [False] * len(all_origins)
    tol = pd.Timedelta(seconds=time_tol)
    rank = {s: i for i, s in enumerate(origin_priority)}
    mag_rank = {s: i for i, s in enumerate(preferred_magnitude_sources)}

    for i in range(len(all_origins)):
        if assigned[i]:
            continue
        event_id += 1
        assigned[i] = True
        members, taken = [i], {srcs[i]}
        for j in range(i + 1, len(all_origins)):
            if times[j] - times[i] > tol:
                break
            if assigned[j] or (one_origin_per_source and srcs[j] in taken):
                continue
            dist = degrees2kilometers(locations2degrees(lats[i], lons[i], lats[j], lons[j]))
            if dist <= dist_tol:
                assigned[j] = True
                members.append(j)
                taken.add(srcs[j])

        group = all_origins.loc[members]
        order = sorted(range(len(group)), key=lambda k: (rank.get(group.iloc[k]['source'], len(rank)), k))
        preferred = group.iloc[order[0]]
        candidates = [k for k in range(len(group)) if group.iloc[k]['source'] in mag_rank
                      and pd.notna(group.iloc[k]['magnitude'])]
        magnitude_row = (group.iloc[min(candidates, key=lambda k: (mag_rank[group.iloc[k]['source']], k))]
                         if candidates else None)
        events.append({
            "event_id": event_id,
            "preferred_origin_id": preferred["origin_id"],
            "preferred_origin_source": preferred["source"],
            "preferred_origin_time": preferred["origin_time"].isoformat(),
            "preferred_latitude": preferred["latitude"],
            "preferred_longitude": preferred["longitude"],
            "preferred_depth_km": preferred["depth_km"],
            "preferred_magnitude": magnitude_row['magnitude'] if magnitude_row is not None else float('nan'),
            "preferred_magnitude_source": magnitude_row['source'] if magnitude_row is not None else None,
            "preferred_magnitude_origin_id": magnitude_row['origin_id'] if magnitude_row is not None else None,
            "origin_selection_policy": '>'.join(origin_priority),
            "magnitude_selection_policy": '>'.join(preferred_magnitude_sources) + ('_nonzero' if zero_magnitude_is_missing else ''),
            "zero_magnitude_is_missing": bool(zero_magnitude_is_missing),
            "n_origins": len(group),
            "sources": '+'.join(sorted(group['source'], key=lambda s: list(sources).index(s))),
        })
        for _, o in group.iterrows():
            origins_out.append({
                "event_id": event_id,
                "origin_id": o["origin_id"],
                "origin_time": o["origin_time"].isoformat(),
                "latitude": o["latitude"],
                "longitude": o["longitude"],
                "depth_km": o["depth_km"],
                "magnitude": o["magnitude"],
                "reported_magnitude": o['reported_magnitude'],
                "magnitude_missing_reason": o['magnitude_missing_reason'],
                "source": o["source"],
                "source_file": o["source_file"],
                "source_line": o["source_line"],
                "rms": o.get("rms"),
                "nass": o.get("nass"),
                "raw_origin_time": o.get("raw_origin_time"),
                "clock_regime": o.get("clock_regime"),
                "clock_uncertainty_s": o.get("clock_uncertainty_s"),
                "note": o.get("note"),
            })

    # ------------------------------------------------------------------
    # Write primary outputs
    # ------------------------------------------------------------------
    out_event, out_origin = Path(out_event_csv), Path(out_origin_csv)
    out_event.parent.mkdir(parents=True, exist_ok=True)
    out_origin.parent.mkdir(parents=True, exist_ok=True)
    df_events = pd.DataFrame(events)
    df_origins = pd.DataFrame(origins_out)
    df_events.to_csv(out_event, index=False)
    df_origins.to_csv(out_origin, index=False)

    # ------------------------------------------------------------------
    # Diagnostics
    # ------------------------------------------------------------------
    comp = df_origins.groupby(["event_id", "source"]).size().unstack(fill_value=0).reset_index()
    for label in sources:
        comp[f"has_{label}"] = comp.get(label, 0) > 0
    combos = df_events['sources'].value_counts().rename_axis('sources').reset_index(name='events')
    preferred_by = df_events['preferred_origin_source'].value_counts()
    magnitude_by = df_events['preferred_magnitude_source'].value_counts(dropna=False)

    print("\nSTEP 43 — HYPOCENTER ASSOCIATION COMPLETE")
    print("----------------------------------------")
    print(f"Total events:            {len(df_events)}")
    print(f"Total origins:           {len(df_origins)}")
    print("\nOrigins by source:", df_origins['source'].value_counts().reindex(list(sources)).to_dict())
    print("Preferred origin by source:", preferred_by.to_dict())
    print("Preferred magnitude by source:", {('none' if pd.isna(k) else k): v for k, v in magnitude_by.items()})
    print("\nSOURCE COMBINATIONS")
    print(combos.to_string(index=False))
    print(f"\nEvent CSV:  {out_event}")
    print(f"Origin CSV: {out_origin}")

    if emit_diagnostics and diagnostics_dir:
        diag_dir = Path(diagnostics_dir)
        diag_dir.mkdir(parents=True, exist_ok=True)
        comp.to_csv(diag_dir / "43_event_source_composition.csv", index=False)
        combos.to_csv(diag_dir / "43_source_combinations.csv", index=False)
        both = comp[comp["has_hypo40"] & comp["has_pinaall"]]
        both.to_csv(diag_dir / "43_events_with_both_sources.csv", index=False)
        comp[comp["has_hypo40"] & ~comp["has_pinaall"]].to_csv(diag_dir / "43_events_hypo40_only.csv", index=False)
        comp[~comp["has_hypo40"] & comp["has_pinaall"]].to_csv(diag_dir / "43_events_pinaall_only.csv", index=False)
        print("\nDiagnostic CSVs written to:")
        print(f"  {diag_dir}")
    return dict(df_events=df_events, df_origins=df_origins, df_composition=comp, df_combinations=combos)

## 3. Execute when ready

This cell reads configured inputs and replaces configured outputs. Inspect summaries and diagnostic tables after the run.

In [ ]:
result = run(**SETTINGS)

In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)
